# 🧹 Module 2: Data Cleaning & Feature Engineering

In this notebook, we perform data preprocessing and cleaning steps:
1. Remove duplicate rows
2. Remove missing `CustomerID` records
3. Remove cancelled invoices (where `InvoiceNo` starts with 'C')
4. Filter out negative or zero `Quantity` and `UnitPrice`
5. Convert `InvoiceDate` to datetime format
6. Create `TotalPrice = Quantity × UnitPrice`
7. Extract temporal features: `Year`, `Month`, `Day`, `Weekday`, `Hour`
8. Save cleaned dataset to `data/processed/cleaned_retail_data.csv`

---

In [1]:
import os
import pandas as pd
import numpy as np

raw_path = os.path.join("..", "data", "raw", "Online Retail.xlsx")
df = pd.read_excel(raw_path)
print(f"Initial Raw Dataset Shape: {df.shape}")

Initial Raw Dataset Shape: (541909, 8)


### Step 1: Remove Duplicate Rows

In [2]:
dup_count = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Removed {dup_count:,} duplicate rows. New Shape: {df.shape}")

Removed 5,268 duplicate rows. New Shape: (536641, 8)


### Step 2: Remove Missing CustomerID

In [3]:
missing_cust = df["CustomerID"].isnull().sum()
df = df.dropna(subset=["CustomerID"])
df["CustomerID"] = df["CustomerID"].astype(int).astype(str)
print(f"Removed {missing_cust:,} records with missing CustomerID. New Shape: {df.shape}")

Removed 135,037 records with missing CustomerID. New Shape: (401604, 8)


### Step 3: Remove Cancelled Orders (InvoiceNo starting with 'C')

In [4]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
cancelled = df["InvoiceNo"].str.upper().str.startswith("C").sum()
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]
print(f"Removed {cancelled:,} cancelled order rows. New Shape: {df.shape}")

Removed 8,872 cancelled order rows. New Shape: (392732, 8)


### Step 4: Remove Quantity <= 0 & UnitPrice <= 0

In [5]:
invalid_qty = (df["Quantity"] <= 0).sum()
invalid_price = (df["UnitPrice"] <= 0).sum()

df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
print(f"Removed {invalid_qty} non-positive quantity rows and {invalid_price} zero-price rows.")
print(f"Cleaned Valid Dataset Shape: {df.shape}")

Removed 0 non-positive quantity rows and 40 zero-price rows.
Cleaned Valid Dataset Shape: (392692, 8)


### Step 5: Feature Engineering (TotalPrice & Datetime Features)

In [6]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

# Extract temporal components
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["Day"] = df["InvoiceDate"].dt.day
df["Weekday"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice,Year,Month,Day,Weekday,Hour
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010,12,1,Wednesday,8
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,12,1,Wednesday,8
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010,12,1,Wednesday,8
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,12,1,Wednesday,8
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010,12,1,Wednesday,8


### Step 6: Export Cleaned Dataset to CSV

In [7]:
out_dir = os.path.join("..", "data", "processed")
os.makedirs(out_dir, exist_ok=True)
out_file = os.path.join(out_dir, "cleaned_retail_data.csv")

df.to_csv(out_file, index=False)
print(f"Cleaned dataset saved successfully to: {out_file}")

Cleaned dataset saved successfully to: ..\data\processed\cleaned_retail_data.csv
